# Notebook 09 — Financial statement modeling

This notebook introduces **financial statement modeling** in finstack using the statements DSL, `ModelBuilder`, and `Evaluator`.

**Prerequisites:** complete the foundation notebooks **01–04** (core types, dates, market data, math) so you are comfortable with periods, numeric data, and running cells top-to-bottom in this series.

## Concept: models as specs + formulas

A **financial model** here is a declarative **specification**: named line items (*nodes*) over a **period lattice** (for example quarterly `2025Q1` … `2025Q4`). Some nodes hold **explicit values** (actuals or assumptions). Others are **computed** from a small **DSL** (e.g. `revenue - cogs`). You can mark where **actuals** end and **forecast** periods begin so downstream tools can treat periods consistently.

Workflow:

1. Configure periods and nodes with `ModelBuilder`.
2. Call `build()` to get an immutable `FinancialModelSpec` (serializable to JSON).
3. Run `Evaluator().evaluate(spec)` to get a `StatementResult` (per-node, per-period numbers), then export to pandas or JSON.

Optional helpers include **formula parse/validate**, **forecast method** tokens (for richer specs), and **normalization** utilities that return JSON or row dicts for adjustment workflows.

### `ModelBuilder`: fluent API

Call `periods(range, actuals_until)` once, then add nodes with `value(...)` (explicit scalars per period) or `compute(...)` (DSL expression). Finish with `build()`.

In [ ]:
import pandas as pd

from finstack.statements import (
    ModelBuilder,
    Evaluator,
    FinancialModelSpec,
    StatementResult,
    parse_formula,
    validate_formula,
    ForecastMethod,
    NodeType,
    NodeId,
    NumericMode,
    NormalizationConfig,
    normalize,
)

b = ModelBuilder("my-pl-model")
b.periods("2025Q1..Q4", "2025Q2")  # Q1–Q2 actuals, Q3–Q4 forecast
b.value(
    "revenue",
    [("2025Q1", 100.0), ("2025Q2", 110.0), ("2025Q3", 115.0), ("2025Q4", 120.0)],
)
b.value(
    "cogs",
    [("2025Q1", 60.0), ("2025Q2", 65.0), ("2025Q3", 68.0), ("2025Q4", 72.0)],
)
b.compute("gross_profit", "revenue - cogs")
b.compute("gross_margin", "gross_profit / revenue")
b.value(
    "opex",
    [("2025Q1", 20.0), ("2025Q2", 22.0), ("2025Q3", 23.0), ("2025Q4", 24.0)],
)
b.compute("ebitda", "gross_profit - opex")
spec = b.build()

print("NodeType.calculated:", NodeType.calculated())
print("NodeId:", NodeId("revenue"))
print("NumericMode.float64:", NumericMode.float64())

print("spec.id:", spec.id)
print("period_count:", spec.period_count)
print("node_count:", spec.node_count)
print("node_ids():", spec.node_ids())
print("has_node(revenue):", spec.has_node("revenue"))
print("schema_version:", spec.schema_version)

### `FinancialModelSpec`: metadata and JSON

The spec is the portable definition of the model. Serialize with `to_json()` and reload with `from_json(...)` for storage, APIs, or scenario tooling.

In [ ]:
json_str = spec.to_json()
print("to_json length:", len(json_str))
print("to_json prefix:", json_str[:180].replace("\n", " "), "...")
spec2 = FinancialModelSpec.from_json(json_str)
print("from_json id:", spec2.id)
print("round-trip node_count match:", spec2.node_count == spec.node_count)

### `Evaluator` and `StatementResult`

`Evaluator().evaluate(spec)` materializes every computed node for every period. `StatementResult` exposes scalars, node slices, counts, timing metadata, JSON round-trip, and **pandas** exports.

In [ ]:
evaluator = Evaluator()
result = evaluator.evaluate(spec)
print(result)
print("sample get:", result.get("revenue", "2025Q1"))
print("ebitda 2025Q4:", result.get("ebitda", "2025Q4"))

### Slicing results and pandas export

Use `get` / `get_node` for ad hoc inspection, or `to_pandas_wide()` / `to_pandas_long()` for analysis. Below, `to_json` / `from_json` checks lossless round-trip for the numeric result set.

In [ ]:
print("get(revenue, 2025Q1):", result.get("revenue", "2025Q1"))
print("get_node(revenue):", result.get_node("revenue"))
print("node_ids():", result.node_ids())
print("node_count:", result.node_count, "num_periods:", result.num_periods)
print("eval_time_ms:", result.eval_time_ms, "warning_count:", result.warning_count)
out_json = result.to_json()
print("result.to_json length:", len(out_json))
result2 = StatementResult.from_json(out_json)
print("round-trip get ebitda Q1:", result2.get("ebitda", "2025Q1"))
wide_df = result.to_pandas_wide()
print(wide_df)
long_df = result.to_pandas_long()
print(long_df)

### DSL: `parse_formula` and `validate_formula`

Computed nodes use the statements expression language. Parse to inspect the compiled form, or validate strings before attaching them to a builder.

In [ ]:
parsed = parse_formula("revenue - cogs")
print("parsed:", parsed)
is_valid = validate_formula("revenue * (1 + growth_rate)")
print("validate_formula:", is_valid)
try:
    validate_formula("revenue +")
except ValueError as err:
    print("invalid formula (expected):", err)

### `ForecastMethod`

Forecast methods are typed tokens used in richer modeling flows (growth curves, forward fill, overrides, etc.). Common factories:

In [ ]:
print("forward_fill:", ForecastMethod.forward_fill())
print("growth_pct:", ForecastMethod.growth_pct())
print("curve_pct:", ForecastMethod.curve_pct())

### `NormalizationConfig` and `normalize`

`normalize(result, config)` returns a **JSON** string describing per-period base values, adjustments, and final values. Parse it with `json.loads` to get Python dict rows; wrap those in pandas for display.

This cell rebuilds a tiny model so it runs even if you skipped earlier sections.

In [ ]:
import json

nb = ModelBuilder("norm-demo")
nb.periods("2025Q1..Q1", None)
nb.value("ebitda", [("2025Q1", 50.0)])
norm_spec = nb.build()
norm_result = Evaluator().evaluate(norm_spec)
config = NormalizationConfig("ebitda")
norm_json = normalize(norm_result, config)
print("normalize JSON:", norm_json)
rows = json.loads(norm_json)
print("rows:", rows)
print(pd.DataFrame(rows))

## Mini-example: quarterly P&L (4 quarters)

Build a compact **P&L**: revenue, COGS, gross profit, OpEx, D&A, EBITDA, EBIT, interest, EBT, taxes (25% of EBT), and net income. Periods `2025Q1`–`2025Q4` with **actuals through `2025Q2`** and forecast in Q3–Q4. Evaluate and print a **pandas wide** table.

In [ ]:
p = ModelBuilder("acme-pl-2025")
p.periods("2025Q1..Q4", "2025Q2")
p.value(
    "revenue",
    [("2025Q1", 100.0), ("2025Q2", 108.0), ("2025Q3", 112.0), ("2025Q4", 118.0)],
)
p.value(
    "cogs",
    [("2025Q1", 55.0), ("2025Q2", 58.0), ("2025Q3", 60.0), ("2025Q4", 63.0)],
)
p.compute("gross_profit", "revenue - cogs")
p.value(
    "opex",
    [("2025Q1", 18.0), ("2025Q2", 19.0), ("2025Q3", 20.0), ("2025Q4", 21.0)],
)
p.value(
    "da",
    [("2025Q1", 4.0), ("2025Q2", 4.0), ("2025Q3", 4.5), ("2025Q4", 5.0)],
)
p.compute("ebitda", "gross_profit - opex")
p.compute("ebit", "ebitda - da")
p.value(
    "interest",
    [("2025Q1", 2.0), ("2025Q2", 2.0), ("2025Q3", 2.2), ("2025Q4", 2.3)],
)
p.compute("ebt", "ebit - interest")
p.compute("taxes", "max(ebt, 0) * 0.25")
p.compute("net_income", "ebt - taxes")
pnl_spec = p.build()
pnl_result = Evaluator().evaluate(pnl_spec)
pnl_wide = pnl_result.to_pandas_wide()
print("P&L wide:")
print(pnl_wide)

## Takeaways

- **`ModelBuilder`** is the ergonomic way to define periods, inputs, and `compute` nodes using the statements **DSL**.
- **`FinancialModelSpec`** is the serializable blueprint (`to_json` / `from_json`), separate from evaluated numbers.
- **`Evaluator`** turns a spec into a **`StatementResult`**, addressable by node and period, exportable to **JSON** or **pandas** (`to_pandas_wide`, `to_pandas_long`).
- **`parse_formula` / `validate_formula`** help you debug and gate formulas before they enter a model.
- **`ForecastMethod`** tokens describe projection styles for more advanced specs.
- **`normalize`** supports EBITDA-style adjustment workflows from an evaluated result; it returns JSON that you can parse with `json.loads` into dict rows.

Run all cells in order when exploring; the walkthrough reuses `spec` and `result`, while the normalization and P&L sections are self-contained.